# 05b Fine-tuning, fitted jointly with last year's volatility

`05-Fine-tuning.ipynb` fine-tuned Yang et al.'s FinBERT on the two-step target of 02 and 03: the straight line first,
then the text on the line's within-year residual. `02b-Joint-ridge.ipynb` found that fitting last year's volatility
$x$ and the text together (with $x$ not penalised) gives much more: the within-year ridge goes from −1.0% to −3.8%
against the line on the paper window, and from −0.4% to −3.2% in 2006. This notebook repeats 05 with that one change.

| What | 05 | Here |
|---|---|---|
| Model | Encoder → average chunk vector → one linear layer | The same, plus $\gamma\,x_{\text{within}}$: one weight on last year's volatility, learnt with the encoder and not weight-decayed |
| Target | The line's within-year residual | Next year's volatility with each year's own mean removed |
| Start | The straight line (the layer starts at zero) | The no-text within-year slope: the layer starts at zero and $\gamma$ at that slope |
| Refit | None: the early-stopped model is the final one | Fine-tuned again from the same seed on 2001 to 2005 for the epoch count that early stopping chose, as 03b and 03c do |
| Everything else | 2006, paper window, 1 seed, up to 3 epochs, 4 chunks per report, early stopping on 2005 | The same |

The prediction is the straight line's average for 2006 plus the within-year part, so the level is exactly the line's,
as in 05. If no epoch beats the no-text slope on 2005, the text is switched off and the prediction is that slope.

**This runs on a Colab GPU.** It reuses the word pieces 05 saved on Drive (`fine-tuning/tokens`), so it does not read
the text again. With `GO = True`, Run all checks the model, then fine-tunes with early stopping (about 1.5 GPU-hours on
a T4) and refits (about another hour), resumable after each epoch of either stage, saving to Google Drive under new
names, so 05's files are untouched. Both versions, without and with the refit, come from the same run.

## 1. Setup

As in notebook 04: on Colab, install pandas 3 (for `data.pkl`), connect Google Drive, get the repo, and link Oscar's
Item 7 text (`sprint2-data/text`) into the places his `Oscar/extract_finbert.py` reads from.

In [1]:
import importlib.metadata, importlib.util, os, shutil, subprocess, sys
from pathlib import Path

NOTEBOOK_DIR = "Alex"
IN_COLAB = importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    # data.pkl was written with pandas 3 and Colab ships pandas 2, which cannot read it.
    if int(importlib.metadata.version("pandas").split(".")[0]) < 3:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pandas==3.0.6"], check=True)
        if "pandas" in sys.modules:
            print("pandas 3 installed but the old one was already loaded: restarting the runtime. Run all again.")
            os.kill(os.getpid(), 9)
    from google.colab import drive
    drive.mount("/content/drive")
    repo = Path("/content/dst-sprint2")
    if repo.exists():
        subprocess.run(["git", "-C", str(repo), "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "https://github.com/Oscar339/dst-sprint2.git", str(repo)], check=True)
    processed = repo / "data" / "processed"
    processed.mkdir(parents=True, exist_ok=True)
    if not (processed / "data.pkl").exists():
        shutil.copy(Path("/content/drive/MyDrive/sprint2-data/data.pkl"), processed / "data.pkl")
    os.chdir(repo / NOTEBOOK_DIR)

Mounted at /content/drive


In [2]:
import copy, math, pickle, platform, time, urllib.request
sys.path.insert(0, "."); sys.path.insert(0, "../Oscar")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display

import scoring

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Python {platform.python_version()} | torch {torch.__version__} | device {DEVICE}"
      + (f" ({torch.cuda.get_device_name(0)})" if DEVICE == "cuda" else ""))

with open("../data/processed/data.pkl", "rb") as f:
    d = pickle.load(f)
df = scoring.frame(d)
x_all, y_all, yr_all = df["logvol_before"].to_numpy(), df["logvol_after"].to_numpy(), df["year"].to_numpy()
print(f"{len(df):,} reports, {df.year.min()}-{df.year.max()}")

DRIVE = Path("/content/drive/MyDrive/sprint2-data")
TEXT_DIR = DRIVE / "text"                      # Oscar's shared folder: edgar_matched/ and item7_source.csv
OUT_DIR = DRIVE / "fine-tuning"                # everything this notebook makes; survives the session
TOK_DIR = OUT_DIR / "tokens"                   # Item 7 word pieces, cut into chunks, one file per year
RESULTS_FILE, HISTORY_FILE = OUT_DIR / "alex-05b-results.csv", OUT_DIR / "alex-05b-history.csv"
TIMING_FILE = OUT_DIR / "alex-05b-timing.csv"   # the word pieces in TOK_DIR are shared with 05

PROCESSED, KOGAN = Path("../data/processed"), Path("../data/raw/kogan")
if IN_COLAB:
    assert (TEXT_DIR / "item7_source.csv").exists() and (TEXT_DIR / "edgar_matched").exists(), \
        f"Oscar's text is not in {TEXT_DIR}: add the shared folder to My Drive as sprint2-data/text"
    shutil.copy(TEXT_DIR / "item7_source.csv", PROCESSED / "item7_source.csv")
    if not (PROCESSED / "edgar_matched").exists():
        (PROCESSED / "edgar_matched").symlink_to(TEXT_DIR / "edgar_matched")
    KOGAN.mkdir(parents=True, exist_ok=True)
    for yr in range(1996, 2007):
        target = KOGAN / f"{yr}.tok.tgz"
        if not target.exists():
            urllib.request.urlretrieve(f"https://www.cs.cmu.edu/~ark/10K/data/{target.name}", target)
    TOK_DIR.mkdir(parents=True, exist_ok=True)
    print("edgar_matched files:", len(list((PROCESSED / "edgar_matched").glob("*.jsonl.gz"))),
          "| Kogan text files:", len(list(KOGAN.glob("*.tok.tgz"))))

Python 3.13.16 | torch 2.14.1+cu130 | device cuda (Tesla T4)
27,142 reports, 1996-2006
edgar_matched files: 11 | Kogan text files: 11


## 2. Settings, the model, and checks before running

**Settings.** Fine-tuning a BERT model is usually done with a small learning rate for the encoder (here $2\times10^{-5}$),
a short warm-up in which the learning rate rises from zero, then a linear decay, for 2 to 4 epochs (Devlin et al.,
2019; Sun et al., 2019). The new layer on top gets a larger learning rate ($10^{-3}$), since it starts from nothing.
The weights are updated by gradient descent on the MSE (lecture 06.1, slides 15 and 16), with AdamW as in notebook
03, and the gradient is clipped so one bad batch cannot throw the encoder far from where it started. Training runs
in mixed precision (Micikevicius et al., 2018): the weights are kept in 32-bit, but most of the arithmetic is done in
16-bit, which is what a T4 is fast at.

**Reading a report.** A report has 15 chunks at the median and up to 64, and every chunk in a training step has to go
through the encoder twice (forwards and backwards). So each training step reads a random 4 of each report's chunks,
averages their vectors and predicts from the average; each epoch draws a new 4. The average of a random 4 is an
unbiased estimate of the average of all of them, so to predict, the model reads every chunk, exactly as the frozen
models did. Averaging is the simplest pooling layer (lecture 06.2, slide 12).

**Starting point.** The new layer starts at zero and $\gamma$ at the no-text within-year slope, so before any training
the model predicts no text effect at all. $\gamma$ is trained with the layer on top (learning rate $10^{-3}$, no weight
decay). Early stopping keeps the epoch with the lowest MSE on the last training year (2005); if no epoch beats the
no-text slope there, the text is switched off.

In [3]:
GO = True          # False: checks only. Set to True once the checks below look right, then Run all again.

from huggingface_hub import hf_hub_download
from tokenizers import BertWordPieceTokenizer
from transformers import BertModel, PreTrainedTokenizerFast
from extract_finbert import CHUNK, MAX_CHUNKS, load_plan, load_texts   # Oscar's chunk size, cap and text rules

TEST_YEAR, WINDOW, SEEDS = 2006, "paper", [0]
FT = dict(chunks_per_report=4,      # chunks read per report in each training step
          micro_reports=2,          # reports per pass through the GPU (8 chunks), to fit in memory
          reports_per_step=16,      # reports per weight update (gradients added over 8 passes)
          max_epochs=3, lr_encoder=2e-5, lr_head=1e-3, weight_decay=0.01, warmup=0.1, clip=1.0,
          dropout=0.1, eval_chunks=32)

YANG, REVISION = "yiyanghkust/finbert-pretrain", "88ab954a39ea6d3ce2b62cff086dd5ad1172c664"
# As in notebook 04: built from the vocabulary, because transformers 5 ignores vocab_file= in BertTokenizerFast.
tok = PreTrainedTokenizerFast(
    tokenizer_object=BertWordPieceTokenizer(hf_hub_download(YANG, "vocab.txt", revision=REVISION), lowercase=True),
    unk_token="[UNK]", sep_token="[SEP]", pad_token="[PAD]", cls_token="[CLS]", mask_token="[MASK]")
tok.model_max_length = 10 ** 9
assert tok.vocab_size == 30_873, "the tokenizer did not load Yang's vocabulary"
CLS_ID, SEP_ID, PAD_ID = tok.cls_token_id, tok.sep_token_id, tok.pad_token_id

def load_encoder():
    # 32-bit weights: they are trained, and mixed precision does the 16-bit arithmetic
    return BertModel.from_pretrained(YANG, revision=REVISION, dtype=torch.float32, add_pooling_layer=False)


class FineTuned(nn.Module):
    # Yang's FinBERT -> token average per chunk -> average over the report's chunks -> one linear layer,
    # plus gamma * x (last year's volatility, within year), learnt with the text
    def __init__(self, encoder, dropout=0.1, gamma0=0.0):
        super().__init__()
        self.encoder = encoder
        self.drop = nn.Dropout(dropout)
        self.head = nn.Linear(encoder.config.hidden_size, 1)
        nn.init.zeros_(self.head.weight); nn.init.zeros_(self.head.bias)    # starts at "no text effect"
        self.gamma = nn.Parameter(torch.tensor(float(gamma0)))

    def chunk_vectors(self, ids, mask):
        with torch.autocast("cuda", dtype=torch.float16, enabled=ids.is_cuda):
            h = self.encoder(input_ids=ids, attention_mask=mask).last_hidden_state
        m = mask.unsqueeze(-1).to(h.dtype)
        return ((h * m).sum(1) / m.sum(1)).float()

    def forward(self, ids, mask, owner, n_reports, xb=None):
        v = self.chunk_vectors(ids, mask)
        pooled = torch.zeros(n_reports, v.shape[1], device=v.device).index_add_(0, owner, v)
        pooled = pooled / torch.bincount(owner, minlength=n_reports).unsqueeze(1).float()
        out = self.head(self.drop(pooled)).squeeze(-1)
        return out if xb is None else out + self.gamma * xb


def chunk_report(text):
    ids = tok(text, add_special_tokens=False)["input_ids"]
    return [np.array(ids[i:i + CHUNK], dtype=np.uint16) for i in range(0, len(ids), CHUNK)][:MAX_CHUNKS]

def pad(chunk_list):
    L = max(len(c) for c in chunk_list) + 2
    ids = torch.full((len(chunk_list), L), PAD_ID, dtype=torch.long)
    mask = torch.zeros((len(chunk_list), L), dtype=torch.long)
    for k, c in enumerate(chunk_list):
        ids[k, :len(c) + 2] = torch.tensor([CLS_ID, *c.tolist(), SEP_ID]); mask[k, :len(c) + 2] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

def train_batch(chunks, idx, rng, k):
    # k random chunks from each report in idx (all of them if it has k or fewer)
    picked, owner = [], []
    for j, i in enumerate(idx):
        for c in rng.choice(len(chunks[i]), size=min(k, len(chunks[i])), replace=False):
            picked.append(chunks[i][c]); owner.append(j)
    ids, mask = pad(picked)
    return ids, mask, torch.tensor(owner, device=DEVICE)

@torch.no_grad()
def predict_reports(model, chunks, idx, batch=32, xb=None):
    # every chunk of every report in idx; returns the model's prediction per report (with gamma * xb if given)
    model.eval()
    flat = [(j, c) for j, i in enumerate(idx) for c in chunks[i]]
    sums = torch.zeros(len(idx), model.head.in_features, device=DEVICE)
    counts = torch.zeros(len(idx), device=DEVICE)
    for s in range(0, len(flat), batch):
        part = flat[s:s + batch]
        owner = torch.tensor([j for j, _ in part], device=DEVICE)
        v = model.chunk_vectors(*pad([c for _, c in part]))
        sums.index_add_(0, owner, v); counts.index_add_(0, owner, torch.ones(len(part), device=DEVICE))
    out = model.head(sums / counts.unsqueeze(1)).squeeze(-1)
    if xb is not None:
        out = out + model.gamma * torch.as_tensor(xb, dtype=torch.float32, device=DEVICE)
    return out.cpu().numpy()

def make_optimizer(model, steps_total):
    decay, no_decay = [], []
    for n, p in model.encoder.named_parameters():
        (no_decay if n.endswith("bias") or "LayerNorm" in n else decay).append(p)
    opt = torch.optim.AdamW([
        {"params": decay, "lr": FT["lr_encoder"], "weight_decay": FT["weight_decay"]},
        {"params": no_decay, "lr": FT["lr_encoder"], "weight_decay": 0.0},
        {"params": [*model.head.parameters(), model.gamma], "lr": FT["lr_head"], "weight_decay": 0.0}])
    warm = max(1, int(FT["warmup"] * steps_total))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warm, max(0.0, (steps_total - s) / max(1, steps_total - warm))))
    return opt, sched

print(f"settings: {FT}")
print(f"test year {TEST_YEAR}, {WINDOW} window: train {scoring.WINDOWS[WINDOW](TEST_YEAR)}, seeds {SEEDS}")

vocab.txt:   0%|          | 0.00/226k [00:00<?, ?B/s]

settings: {'chunks_per_report': 4, 'micro_reports': 2, 'reports_per_step': 16, 'max_epochs': 3, 'lr_encoder': 2e-05, 'lr_head': 0.001, 'weight_decay': 0.01, 'warmup': 0.1, 'clip': 1.0, 'dropout': 0.1, 'eval_chunks': 32}
test year 2006, paper window: train range(2001, 2006), seeds [0]


**Checks.** On 32 reports from 2006: that the untrained model predicts exactly zero (the straight line), that a few
training steps change the encoder's weights and bring the loss down on those reports, how much GPU memory a step
takes, and how fast training and prediction run. The timings give the estimate for the full run. The checks train a
copy of the model, which is then thrown away.

In [4]:
plan = load_plan()
assert set(plan["key"]) == set(df["key"]), "the reports must be exactly the reports in data.pkl"
key_row = {k: i for i, k in enumerate(df["key"])}
rows_check = plan[plan["year"] == 2006].head(32)
texts_check = load_texts(2006, rows_check)
check_chunks = {key_row[k]: chunk_report(texts_check[k][1]) for k in rows_check["key"]}
check_idx = np.array(list(check_chunks))
z = (y_all - y_all.mean()) / y_all.std()                    # any target will do to see the loss fall

torch.manual_seed(0)
model = FineTuned(load_encoder(), FT["dropout"]).to(DEVICE)
print(f"encoder: {sum(p.numel() for p in model.encoder.parameters()) / 1e6:.0f}M parameters, all trained")
p0 = predict_reports(model, check_chunks, check_idx[:4])
assert np.all(p0 == 0), "the untrained model should predict exactly zero"
print("untrained prediction is exactly zero: the model starts at the straight line")

# a bias: weight decay does not touch it, so it only changes if a gradient reaches the encoder
before = model.encoder.encoder.layer[0].attention.self.query.bias.detach().clone()
opt, sched = make_optimizer(model, 100)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE == "cuda")
rng = np.random.default_rng(0)
fixed = check_idx[:FT["micro_reports"]]
if DEVICE == "cuda":
    torch.cuda.reset_peak_memory_stats()
losses, n_chunks, t_train = [], 0, 0.0
for step in range(24):
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.train()
    ids, mask, owner = train_batch(check_chunks, fixed, rng, FT["chunks_per_report"])
    opt.zero_grad(set_to_none=True)
    loss = ((model(ids, mask, owner, len(fixed)) - torch.tensor(z[fixed], dtype=torch.float32, device=DEVICE)) ** 2).mean()
    scaler.scale(loss).backward()
    scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), FT["clip"])
    scaler.step(opt); scaler.update(); sched.step()
    if DEVICE == "cuda":
        torch.cuda.synchronize()
    if step >= 4:                                            # the first steps include warm-up costs
        t_train += time.perf_counter() - t0; n_chunks += len(ids)
    losses.append(loss.item())
changed = (model.encoder.encoder.layer[0].attention.self.query.bias - before).abs().max().item()
train_rate = n_chunks / t_train
peak = torch.cuda.max_memory_allocated() / 2 ** 30 if DEVICE == "cuda" else float("nan")
print(f"loss on 2 reports: {np.mean(losses[:3]):.3f} at the start -> {np.mean(losses[-3:]):.3f} after 24 steps; "
      f"largest change to a first-layer bias {changed:.1e}")
assert changed > 0, "the encoder's weights did not change: it is not being trained"

if DEVICE == "cuda":
    torch.cuda.synchronize()
t0 = time.perf_counter()
predict_reports(model, check_chunks, check_idx, FT["eval_chunks"])
if DEVICE == "cuda":
    torch.cuda.synchronize()
eval_rate = sum(len(c) for c in check_chunks.values()) / (time.perf_counter() - t0)
print(f"training {train_rate:.0f} chunks/s | predicting {eval_rate:.0f} chunks/s | peak GPU memory {peak:.1f} GB")
del model, opt, sched, scaler
if DEVICE == "cuda":
    torch.cuda.empty_cache()

# Estimate the full run from the first encoder's chunk counts, scaled by Yang's 4% fewer word pieces (notebook 04)
train_mask, test_mask = scoring.masks(df, TEST_YEAR, WINDOW)
years = scoring.WINDOWS[WINDOW](TEST_YEAR)
n_chunks_all = np.array([len(c) for c in d["X_item7"]]) * 480_332 / 498_607
fit_mask, val_mask = train_mask & (yr_all != years[-1]), train_mask & (yr_all == years[-1])
per_epoch = (np.minimum(n_chunks_all[fit_mask], FT["chunks_per_report"]).sum() / train_rate
             + n_chunks_all[val_mask].sum() / eval_rate)
total = FT["max_epochs"] * per_epoch + n_chunks_all[test_mask].sum() / eval_rate
print(f"fit {fit_mask.sum():,} reports, early stopping {val_mask.sum():,}, test {test_mask.sum():,}: "
      f"about {per_epoch / 60:.0f} min per epoch, {total / 3600:.1f} hours for {FT['max_epochs']} epochs per seed")
timing = pd.DataFrame([{"device": DEVICE, "train_chunks_per_second": train_rate, "predict_chunks_per_second": eval_rate,
                        "peak_memory_gb": peak, "minutes_per_epoch_estimate": per_epoch / 60}])
if IN_COLAB:
    timing.to_csv(TIMING_FILE, index=False)
if not GO:
    print("\nGO is False: checks done, nothing fine-tuned. Set GO = True above and Run all again.")

config.json:   0%|          | 0.00/359 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  442MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  442MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


encoder: 109M parameters, all trained
untrained prediction is exactly zero: the model starts at the straight line
loss on 2 reports: 0.267 at the start -> 0.004 after 24 steps; largest change to a first-layer bias 2.2e-04
training 37 chunks/s | predicting 159 chunks/s | peak GPU memory 3.0 GB
fit 11,271 reports, early stopping 3,121, test 3,003: about 28 min per epoch, 1.5 hours for 3 epochs per seed


## 3. The text

Item 7 for the six years needed (2001 to 2006), cut into chunks of 510 word pieces, at most 64 per report, with
Oscar's rules for which text to read, exactly as in notebook 04. The word pieces are saved to Drive one year at a
time, so a rerun does not read the text again.

In [5]:
CHUNKS = None
if GO and IN_COLAB and DEVICE == "cuda":
    CHUNKS = [None] * len(df)
    for yr in sorted(set(years) | {TEST_YEAR}):
        path = TOK_DIR / f"{yr}.npz"
        if not path.exists():
            rows_y = plan[plan["year"] == yr]
            texts = load_texts(yr, rows_y)
            per = [chunk_report(texts[k][1]) for k in rows_y["key"]]
            tmp = TOK_DIR / f"{yr}.tmp.npz"
            np.savez(tmp, keys=np.array(rows_y["key"]), n=np.array([len(p) for p in per]),
                     lens=np.array([len(c) for p in per for c in p]),
                     ids=np.concatenate([c for p in per for c in p]))
            os.replace(tmp, path)
        zf = np.load(path, allow_pickle=True)
        pieces = np.split(zf["ids"], np.cumsum(zf["lens"])[:-1])
        starts = np.concatenate([[0], np.cumsum(zf["n"])[:-1]])
        for k, s0, n in zip(zf["keys"], starts, zf["n"]):
            CHUNKS[key_row[str(k)]] = pieces[s0:s0 + n]
        print(f"{yr}: {len(zf['keys']):,} reports, {zf['n'].sum():,} chunks", flush=True)
    needed = np.flatnonzero(train_mask | test_mask)
    assert all(CHUNKS[i] is not None and len(CHUNKS[i]) > 0 for i in needed), "every report needs Item 7 chunks"
else:
    print("skipped (GO is False, or not on a Colab GPU)")

2001: 2,299 reports, 36,202 chunks
2002: 2,587 reports, 49,898 chunks
2003: 3,217 reports, 69,172 chunks
2004: 3,168 reports, 73,724 chunks
2005: 3,121 reports, 70,266 chunks
2006: 3,003 reports, 67,746 chunks


## 4. Fine-tuning

The target is next year's volatility with each training year's own mean removed, and $x$ the same for last year's,
both scaled as in `joint.py` (the target by the spread left after the no-text slope, so the layer on top learns on
the same scale as in 05). After each epoch, the model is scored on 2005, and the model, the optimiser and the history
are saved to Drive, so a session that stops carries on from the last finished epoch. The random chunks drawn in each
epoch depend only on the seed and the epoch, so a resumed run draws the same ones.

**Refit.** Once early stopping has chosen the number of epochs, the model is fine-tuned again from the same seed on
all five training years (2001 to 2005) for that many epochs, with the target and $x$ rescaled on those years and
the same learning-rate schedule per epoch, and no scoring on 2005, which is now training data. If early stopping
switched the text off, the refit is the no-text slope on the five years. This is the rule of 03b and 03c.

In [ ]:
def fine_tune(seed, fit_idx, val_idx, target, tag, xs, gamma0, n_epochs=None):
    """Fine-tune on fit_idx. With val_idx: up to max_epochs, keep the best epoch on val_idx (or None if no epoch beats
    the no-text slope). Without: exactly n_epochs, keep the last. The learning-rate schedule always spans max_epochs,
    so a refit for n epochs follows the same schedule as the first n epochs of the early-stopped run.
    Returns (model or None, history, best epoch, 0-based or -1)."""
    stage = "no refit" if val_idx is not None else "refit"
    epochs = FT["max_epochs"] if val_idx is not None else n_epochs
    ckpt, best_path = OUT_DIR / f"{tag}_checkpoint.pt", OUT_DIR / f"{tag}_best.pt"
    torch.manual_seed(seed)
    model = FineTuned(load_encoder(), FT["dropout"], gamma0).to(DEVICE)
    steps_per_epoch = math.ceil(len(fit_idx) / FT["reports_per_step"])
    opt, sched = make_optimizer(model, steps_per_epoch * FT["max_epochs"])
    scaler = torch.amp.GradScaler("cuda")
    text_off = float(np.mean((target[val_idx] - gamma0 * xs[val_idx]) ** 2)) if val_idx is not None else np.nan
    history, best, start = [], (text_off, -1), 0                                # epoch -1: text switched off
    if ckpt.exists():
        state = torch.load(ckpt, map_location=DEVICE, weights_only=False)
        model.load_state_dict(state["model"]); opt.load_state_dict(state["opt"])
        sched.load_state_dict(state["sched"]); scaler.load_state_dict(state["scaler"])
        history, best, start = state["history"], state["best"], state["epoch"]
        print(f"{stage}: resuming after epoch {start}")
    t_all = torch.tensor(target, dtype=torch.float32, device=DEVICE)
    x_t = torch.tensor(xs, dtype=torch.float32, device=DEVICE)
    for epoch in range(start, epochs):
        rng = np.random.default_rng([seed, epoch])
        torch.manual_seed(1000 * seed + epoch)
        order = rng.permutation(fit_idx)
        model.train()
        t0, total, n_chunks = time.perf_counter(), 0.0, 0
        for step, s in enumerate(range(0, len(order), FT["reports_per_step"])):
            step_idx = order[s:s + FT["reports_per_step"]]
            opt.zero_grad(set_to_none=True)
            for m0 in range(0, len(step_idx), FT["micro_reports"]):
                b = step_idx[m0:m0 + FT["micro_reports"]]
                ids, mask, owner = train_batch(CHUNKS, b, rng, FT["chunks_per_report"])
                bt = torch.as_tensor(b, device=DEVICE)
                pred = model(ids, mask, owner, len(b), x_t[bt])
                loss = ((pred - t_all[bt]) ** 2).sum() / len(step_idx)
                scaler.scale(loss).backward()
                total += loss.item() * len(step_idx); n_chunks += len(ids)
            scaler.unscale_(opt); nn.utils.clip_grad_norm_(model.parameters(), FT["clip"])
            scaler.step(opt); scaler.update(); sched.step()
            if step % 100 == 0:
                print(f"  {stage} epoch {epoch + 1} step {step}/{steps_per_epoch}: "
                      f"{n_chunks / (time.perf_counter() - t0):.0f} chunks/s", flush=True)
        train_seconds = time.perf_counter() - t0
        t0 = time.perf_counter()
        row = {"seed": seed, "stage": stage, "epoch": epoch + 1, "train_mse": total / len(fit_idx),
               "val_mse": np.nan, "val_mse_text_off": text_off, "gamma": float(model.gamma.detach()),
               "train_seconds": train_seconds, "train_chunks": n_chunks, "val_seconds": 0.0}
        if val_idx is not None:
            val_mse = float(np.mean((predict_reports(model, CHUNKS, val_idx, FT["eval_chunks"], xs[val_idx]) - target[val_idx]) ** 2))
            row.update(val_mse=val_mse, val_seconds=time.perf_counter() - t0)
            if val_mse < best[0]:
                best = (val_mse, epoch)
                torch.save(model.state_dict(), OUT_DIR / f"{tag}_best.tmp.pt"); os.replace(OUT_DIR / f"{tag}_best.tmp.pt", best_path)
        history.append(row)
        print(f"{stage} epoch {epoch + 1}: train MSE {row['train_mse']:.4f}, 2005 MSE {row['val_mse']:.4f} "
              f"(text off: {text_off:.4f}), {train_seconds / 60:.0f} min", flush=True)
        torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "history": history, "best": best, "epoch": epoch + 1},
                   OUT_DIR / f"{tag}_checkpoint.tmp.pt")
        os.replace(OUT_DIR / f"{tag}_checkpoint.tmp.pt", ckpt)
    if val_idx is None:
        print(f"refit: {epochs} epochs on {len(fit_idx):,} reports")
        return model, history, epochs - 1
    if best[1] < 0:
        print("no epoch beat the no-text slope on 2005: text switched off")
        return None, history, -1
    model.load_state_dict(torch.load(best_path, map_location=DEVICE))
    print(f"kept epoch {best[1] + 1} (2005 MSE {best[0]:.4f})")
    return model, history, best[1]


def centre_years(v, idx):
    """v with each year's own mean removed, over the rows idx (zero elsewhere)."""
    out = np.zeros(len(df))
    out[idx] = pd.Series(v[idx]).groupby(yr_all[idx]).transform(lambda s: s - s.mean()).to_numpy()
    return out


def scaled(train_idx, test_idx, idx):
    """The joint target and x of joint.py: each year's own mean removed, scaled on the rows idx."""
    yc = centre_years(y_all, train_idx)
    xc = centre_years(x_all, train_idx) + centre_years(x_all, test_idx)
    g = (xc[idx] @ yc[idx]) / (xc[idx] @ xc[idx])                  # the no-text within-year slope
    scale, sx = (yc[idx] - g * xc[idx]).std(), xc[idx].std()
    return yc / scale, xc / sx, g * sx / scale, scale, sx


FITS = {}

def fit_both(seed, train, test, history_log):
    """Early-stopped and refitted joint fine-tuning for one split; both test predictions, cached."""
    key = (seed, int(yr_all[test][0]))
    if key in FITS:
        return FITS[key]
    yrs = sorted(set(yr_all[train]))
    fit, val = train & (yr_all != yrs[-1]), train & (yr_all == yrs[-1])
    fit_idx, val_idx, train_idx, test_idx = (np.flatnonzero(m) for m in (fit, val, train, test))
    a, b = scoring.fit_line(df, train)
    line_level = np.mean(a + b * x_all[test_idx])
    out = {}

    def finish(model, xs, gamma0, scale, sx, version):
        if model is None:
            z = gamma0 * xs[test_idx]
        else:
            t0 = time.perf_counter()
            z = predict_reports(model, CHUNKS, test_idx, FT["eval_chunks"], xs[test_idx])
            print(f"{version}: predicted {len(test_idx):,} test reports in {(time.perf_counter() - t0) / 60:.0f} min; "
                  f"slope on x: start {gamma0 * scale / sx:.3f}, learnt {float(model.gamma.detach()) * scale / sx:.3f}, "
                  f"line {b:.3f}")
        z = z * scale
        out[version] = line_level + z - z.mean()                         # centred on the test year

    # 1. early stopping on the last training year (as in 05)
    target, xs, gamma0, scale, sx = scaled(train_idx, test_idx, fit_idx)
    model, history, best = fine_tune(seed, fit_idx, val_idx, target, f"yang_joint_{WINDOW}_{TEST_YEAR}_seed{seed}", xs, gamma0)
    history_log.extend(history)
    finish(model, xs, gamma0, scale, sx, "no refit")
    del model
    torch.cuda.empty_cache()

    # 2. refit on the whole window for the chosen number of epochs
    target, xs, gamma0, scale, sx = scaled(train_idx, test_idx, train_idx)
    if best < 0:
        model = None                                                     # text switched off: the no-text slope
    else:
        model, history, _ = fine_tune(seed, train_idx, None, target, f"yang_joint_refit_{WINDOW}_{TEST_YEAR}_seed{seed}",
                                      xs, gamma0, n_epochs=best + 1)
        history_log.extend(history)
    finish(model, xs, gamma0, scale, sx, "refit")
    del model
    torch.cuda.empty_cache()
    FITS[key] = out
    return out


if CHUNKS is not None:
    done = pd.read_csv(RESULTS_FILE)["model"].tolist() if RESULTS_FILE.exists() else []
    for seed in SEEDS:
        names = {v: f"joint fine-tuned Yang, Item 7, {v}, seed {seed}" for v in ["no refit", "refit"]}
        if all(n in done for n in names.values()):
            continue
        log = []
        res = pd.concat([scoring.evaluate(df, (lambda df_, tr, te, v=v: fit_both(seed, tr, te, log)[v]), name=n,
                                          windows=[WINDOW], test_years=[TEST_YEAR]) for v, n in names.items()],
                        ignore_index=True)
        res.to_csv(RESULTS_FILE, mode="a", header=not RESULTS_FILE.exists(), index=False)
        pd.DataFrame(log).to_csv(HISTORY_FILE, mode="a", header=not HISTORY_FILE.exists(), index=False)
    for f in [RESULTS_FILE, HISTORY_FILE, TIMING_FILE]:
        if f.exists():
            shutil.copy(f, Path("../data/output") / f.name)
    print("done; result files copied to data/output/")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  no refit epoch 1 step 0/705: 46 chunks/s
  no refit epoch 1 step 100/705: 40 chunks/s
  no refit epoch 1 step 200/705: 40 chunks/s
  no refit epoch 1 step 300/705: 40 chunks/s
  no refit epoch 1 step 400/705: 40 chunks/s
  no refit epoch 1 step 500/705: 40 chunks/s
  no refit epoch 1 step 600/705: 40 chunks/s
  no refit epoch 1 step 700/705: 40 chunks/s
no refit epoch 1: train MSE 0.9993, 2005 MSE 0.8162 (text off: 0.8251), 19 min
  no refit epoch 2 step 0/705: 39 chunks/s
  no refit epoch 2 step 100/705: 39 chunks/s
  no refit epoch 2 step 200/705: 39 chunks/s
  no refit epoch 2 step 300/705: 39 chunks/s
  no refit epoch 2 step 400/705: 40 chunks/s
  no refit epoch 2 step 500/705: 40 chunks/s
  no refit epoch 2 step 600/705: 40 chunks/s
  no refit epoch 2 step 700/705: 40 chunks/s
no refit epoch 2: train MSE 0.9581, 2005 MSE 0.7944 (text off: 0.8251), 19 min
  no refit epoch 3 step 0/705: 39 chunks/s
  no refit epoch 3 step 100/705: 39 chunks/s
  no refit epoch 3 step 200/705: 39 ch

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: yiyanghkust/finbert-pretrain
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  refit epoch 1 step 0/900: 41 chunks/s
  refit epoch 1 step 100/900: 40 chunks/s
  refit epoch 1 step 200/900: 40 chunks/s
  refit epoch 1 step 300/900: 40 chunks/s
  refit epoch 1 step 400/900: 40 chunks/s
  refit epoch 1 step 500/900: 40 chunks/s
  refit epoch 1 step 600/900: 40 chunks/s
  refit epoch 1 step 700/900: 40 chunks/s
  refit epoch 1 step 800/900: 40 chunks/s
refit epoch 1: train MSE 1.0044, 2005 MSE nan (text off: nan), 24 min
  refit epoch 2 step 0/900: 40 chunks/s
  refit epoch 2 step 100/900: 39 chunks/s
  refit epoch 2 step 200/900: 39 chunks/s
  refit epoch 2 step 300/900: 39 chunks/s


## 5. Results

Against 05 and the joint frozen models on the same test year, window and reports, as the change in MSE against the
straight line (negative is better), with the paired bootstrap interval and the spread part. Rows appear as their files
are in `data/output/` (02b; 03c for the joint attention network and the joint ridge on Yang's vectors).

In [ ]:
out = Path("../data/output")
if (out / "alex-05b-results.csv").exists():
    files = {"alex-05b-results.csv": None, "alex-05-results.csv": None,
             "alex-02b-joint-ridge-results.csv": ("two-step within-year ridge, Item 7", "joint within-year ridge, Item 7"),
             "alex-03c-yang-joint-ridge-results.csv": None,
             "alex-03c-joint-attention-results.csv": ("joint attention, Item 7, refit",)}
    rows = []
    for f, keep in files.items():
        if (out / f).exists():
            t = pd.read_csv(out / f)
            if keep is not None:
                t = t[t["model"].str.startswith(keep)]
            rows.append(t)
    rows = pd.concat(rows)
    rows["model"] = rows["model"].str.replace(r", seed \d+$", "", regex=True)
    rows = rows[(rows["window"] == WINDOW) & (rows["test_year"] == TEST_YEAR)]
    cols = ["mse", "diff", "ci_low", "ci_high", "change_pct", "spread_change_pct"]
    table = rows.groupby("model", sort=False)[cols].mean()
    display(table.style.format({"mse": "{:.4f}", "diff": "{:+.4f}", "ci_low": "{:+.4f}", "ci_high": "{:+.4f}",
                                "change_pct": "{:+.2f}%", "spread_change_pct": "{:+.2f}%"}))
    print(f"straight line MSE in {TEST_YEAR}: {rows['baseline_mse'].iloc[0]:.4f}")
else:
    print("results not there yet")

**Training curve.** For the early-stopping stage, the MSE on the training reports and on 2005 after each epoch, in units of the target's standard
deviation; the dashed line is 2005 with the text switched off (the no-text slope). Below the dashed line, the
fine-tuned text helps on the year used for early stopping.

In [ ]:
if (out / "alex-05b-history.csv").exists():
    hist_all = pd.read_csv(out / "alex-05b-history.csv")
    display(hist_all.round(4))
    hist = hist_all[hist_all["stage"] == "no refit"]
    fig, ax = plt.subplots(figsize=(6.5, 3.8))
    for seed, h in hist.groupby("seed"):
        ax.plot(h["epoch"], h["train_mse"], "o-", color="#2a78d6", label=f"training reports (2001-2004), seed {seed}")
        ax.plot(h["epoch"], h["val_mse"], "s-", color="#eb6834", label=f"2005 (early stopping), seed {seed}")
    ax.axhline(hist["val_mse_text_off"].iloc[0], color="#52514e", ls="--", lw=1, label="2005, text switched off")
    ax.set_xlabel("epoch"); ax.set_ylabel("MSE (standardised residual)")
    ax.set_xticks(sorted(hist["epoch"].unique()))
    ax.grid(axis="y", color="#e6e5e0", lw=0.8); ax.spines[["top", "right"]].set_visible(False)
    ax.legend(frameon=False, fontsize=8)
    ax.set_title("Figure 1. Joint fine-tuning of Yang's FinBERT, test year 2006", fontsize=11, loc="left")
    fig.tight_layout(); plt.show()
    print(f"training: {hist_all['train_chunks'].sum() / hist_all['train_seconds'].sum():.0f} chunks/s, "
          f"{(hist_all['train_seconds'] + hist_all['val_seconds']).sum() / 3600:.2f} GPU-hours for {len(hist_all)} epochs "
          f"(both stages, without the test predictions)")

## 6. What this shows

*To be written once the run is done.*

## References

- Devlin, J., Chang, M.-W., Lee, K. & Toutanova, K. (2019). BERT: Pre-training of deep bidirectional transformers for language understanding. *NAACL-HLT 2019*. https://aclanthology.org/N19-1423/
- Loshchilov, I. & Hutter, F. (2019). Decoupled weight decay regularization. *ICLR 2019*.
- Micikevicius, P. et al. (2018). Mixed precision training. *ICLR 2018*.
- Sun, C., Qiu, X., Xu, Y. & Huang, X. (2019). How to fine-tune BERT for text classification? *CCL 2019*. arXiv:1905.05583
- Xia, B., Rawte, V. D., Zaki, M. J. & Gupta, A. (2022). FETILDA: An effective framework for fin-tuned embeddings for long financial text documents. *arXiv:2206.06952*.
- Yang, Y., Uy, M. C. S. & Huang, A. (2020). FinBERT: A pretrained language model for financial communications. *arXiv:2006.08097*. Model: https://huggingface.co/yiyanghkust/finbert-pretrain

Course material (Data Science Toolbox, University of Bristol, 2026):

- Assessment 2 brief (pre-trained models, transfer learning, fine-tuning). https://dsbristol.github.io/dst/assets/assessments/Assessment2.pdf
- Lecture 05.2, Statistical Testing. https://dsbristol.github.io/dst/assets/slides/05.2-StatisticalTesting.pdf
- Lecture 06.1, Neural Networks 1 (loss functions, gradient descent). https://dsbristol.github.io/dst/assets/slides/06.1-Neural%20Networks%20Lecture%201.pdf
- Lecture 06.2, Neural Networks 2 (pooling layers). https://dsbristol.github.io/dst/assets/slides/06.2-Neural%20Networks%20Lecture%202.pdf
- Lecture 09.1, Introduction to Parallelism. https://dsbristol.github.io/dst/assets/slides/09.1-ParallelAlgorithms.pdf
- Lecture 09.2, Mapping, Reducing, and Computational Graphs. https://dsbristol.github.io/dst/assets/slides/09.2-ComputationalGraphs.pdf